# Tutorial 8: Three-Model Coupling

Estimated time: 25-45 minutes

## Prerequisites
`pymc` recommended.

## Learning aims
- Extend the metamodel to three coupled surrogates (chain or fan-out)
- Read posterior diagnostics and validate against held-out runs


In [ ]:
# Cross-platform setup — works on Windows / macOS / Linux.
# Self-contained: walks up to find the repo root, adds src/ to sys.path,
# then imports the official bootstrap helper for chdir + later run_cli use.
import os, sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    _marker = _candidate / 'pyproject.toml'
    if _marker.is_file() and 'name = "metamodeler"' in _marker.read_text():
        ROOT = _candidate
        break
else:
    raise FileNotFoundError('Could not locate metamodeler repo root from ' + str(_here))

_src = str((ROOT / 'src').resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
if Path.cwd().resolve() != ROOT.resolve():
    os.chdir(ROOT)

from metamodeler.tutorial import bootstrap, run_cli  # noqa: E402
ROOT = bootstrap()
print('Repo root:', ROOT)


## Step 1: Build IR and sample the three-model coupled metamodel

In [ ]:
spec_path = 'tutorials/specs/metamodel.three_model.pymc.json'
run_cli('meta', 'build', spec_path)
run_cli(
    'meta', 'sample', spec_path,
    '--draws', '120', '--tune', '60', '--chains', '2', '--seed', '2',
)


## Step 2: Inspect IR + samples

In [ ]:
run_cli('meta', 'list')


## Step 3: Plot the joint posterior marginals

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

samples_root = ROOT / 'tmp/metamodel_samples'
if not samples_root.is_dir():
    print('No metamodel samples yet.')
else:
    latest = sorted(p for p in samples_root.iterdir() if p.is_dir())[-1]
    payload = json.loads((latest / 'samples_dataset.json').read_text())
    n_vars = len(payload['variables'])
    fig, axes = plt.subplots(1, n_vars, figsize=(2.5 * n_vars, 3))
    if n_vars == 1:
        axes = [axes]
    for ax, (name, values) in zip(axes, payload['variables'].items()):
        flat = np.array(values, dtype=float).reshape(-1)
        ax.hist(flat, bins=30)
        ax.set_title(name)
    plt.tight_layout()
    plt.show()
